# 02. Feature Engineering

Извлечение и эксперименты с различными типами признаков.

In [ ]:
import os
import pandas as pd
import numpy as np
import re
from urllib.parse import urlparse
from sklearn.feature_extraction.text import CountVectorizer
from gensim.models import Word2Vec
import warnings
warnings.filterwarnings('ignore')

## Загрузка данных

In [ ]:
df = pd.read_csv('raw_data/CEAS_08.csv')
print(f"Dataset shape: {df.shape}")

## Признаки отправителя

In [ ]:
def extract_sender_features(df):
    FREE_EMAIL_DOMAINS = ['gmail.com', 'yahoo.com', 'hotmail.com', 'outlook.com', 'mail.ru', 'yandex.ru']
    
    def extract_domain(sender):
        if pd.isna(sender):
            return ''
        match = re.search(r'@([\w.-]+)', sender)
        return match.group(1) if match else ''
    
    df['sender_domain'] = df['sender'].apply(extract_domain)
    df['is_free_email'] = df['sender_domain'].apply(lambda x: 1 if x in FREE_EMAIL_DOMAINS else 0)
    df['domain_length'] = df['sender_domain'].apply(len)
    df['has_numbers_in_domain'] = df['sender_domain'].apply(lambda x: int(bool(re.search(r'\d', x))))
    
    return df

df = extract_sender_features(df)
print("Sender features created")
df[['sender', 'sender_domain', 'is_free_email', 'domain_length']].head()

## Признаки URL

In [ ]:
def extract_urls(text):
    return re.findall(r'http\S+|www\.\S+', str(text))

def url_features(df):
    df['url_count'] = df['body'].apply(lambda x: len(extract_urls(x)))
    df['avg_url_length'] = df['body'].apply(
        lambda x: np.mean([len(u) for u in extract_urls(x)]) if extract_urls(x) else 0
    )
    df['has_suspicious_tld'] = df['body'].apply(
        lambda x: int(any(u.endswith(('.xyz', '.top', '.click', '.work')) for u in extract_urls(x)))
    )
    df['has_ip_url'] = df['body'].apply(
        lambda x: int(bool(re.search(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', str(x))))
    )
    return df

df = url_features(df)
print("URL features created")
df[['url_count', 'avg_url_length', 'has_suspicious_tld', 'has_ip_url']].head()

## Текстовые признаки (NLP)

In [ ]:
URGENCY_WORDS = ['urgent', 'immediately', 'now', 'act now', 'limited time', 'expire', 'deadline']
THREAT_WORDS = ['suspended', 'blocked', 'account', 'unauthorized', 'security', 'alert', 'warning']
REWARD_WORDS = ['won', 'prize', 'gift', 'free', 'bonus', 'congratulations', 'winner']

def text_nlp_features(df):
    df['urgency_count'] = df['body'].apply(
        lambda x: sum(1 for w in URGENCY_WORDS if w in str(x).lower())
    )
    df['threat_count'] = df['body'].apply(
        lambda x: sum(1 for w in THREAT_WORDS if w in str(x).lower())
    )
    df['reward_count'] = df['body'].apply(
        lambda x: sum(1 for w in REWARD_WORDS if w in str(x).lower())
    )
    df['capital_count'] = df['body'].apply(lambda x: sum(1 for c in str(x) if c.isupper()))
    df['exclamation_count'] = df['body'].apply(lambda x: str(x).count('!'))
    df['text_length'] = df['body'].fillna('').apply(len)
    df['word_count'] = df['body'].fillna('').apply(lambda x: len(x.split()))
    return df

df = text_nlp_features(df)
print("Text NLP features created")
df[['urgency_count', 'threat_count', 'reward_count', 'text_length', 'word_count']].head()

## TF-IDF Features

In [ ]:
df['text'] = df['subject'].fillna('') + ' ' + df['body'].fillna('')

print(f"Text column shape: {df['text'].shape}")
print("TF-IDF will be fitted later on the training split to avoid leakage.")

## Объединение всех признаков

In [ ]:
numeric_cols = [
    'is_free_email', 'domain_length', 'has_numbers_in_domain',
    'url_count', 'avg_url_length', 'has_suspicious_tld', 'has_ip_url',
    'urgency_count', 'threat_count', 'reward_count', 
    'capital_count', 'exclamation_count', 'text_length', 'word_count'
]

manual_features_df = df[numeric_cols].reset_index(drop=True)
text_data = df['text'].reset_index(drop=True)
labels = df['label'].reset_index(drop=True)

print(f"Manual features shape: {manual_features_df.shape}")
print(f"Text data shape: {text_data.shape}")
print(f"Labels shape: {labels.shape}")

## Сохранение признаков

In [ ]:
os.makedirs('features_data', exist_ok=True)
manual_features_df.to_csv('features_data/manual_features.csv', index=False)
text_data.to_frame(name='text').to_csv('features_data/text_data.csv', index=False)
labels.to_frame(name='label').to_csv('features_data/labels.csv', index=False)
print("Manual features, text data, and labels saved to features_data/")